# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal check 1 — staleness

claim: stale-but-visible pages (days_since_last_update >= 180, impressions_90d >= 500 — same thresholds the real `stale_visible_page` flag uses) show weaker current performance than fresh visible pages.


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/mrym828/Machine-learning-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(df.shape)

visible = df[df['impressions_90d'] >= 500].copy()
visible['is_stale'] = visible['days_since_last_update'] >= 180

bucket_staleness = visible.groupby('is_stale').agg(
    n=('content_id', 'count'),
    mean_ctr=('ctr', 'mean'),
    mean_engagement_rate=('engagement_rate', 'mean'),
).reset_index()
bucket_staleness 

(30000, 44)


,is_stale,n,mean_ctr,mean_engagement_rate
0,False,16709,0.262262,3.041436
1,True,17,0.208824,5.338824


In [2]:
tier_order = ['never', '0-30', '31-90', '91-180', '181+']

bucket_freshness_tier = visible.groupby('freshness_tier').agg(
    n=('content_id', 'count'),
    mean_ctr=('ctr', 'mean'),
    mean_engagement_rate=('engagement_rate', 'mean'),
).reindex(tier_order).dropna(how='all')

bucket_freshness_tier


,n,mean_ctr,mean_engagement_rate
freshness_tier,,,
0-30,10063.0,0.271484,3.314373
31-90,88.0,0.144886,2.553636
91-180,6558.0,0.249686,2.629169
181+,17.0,0.208824,5.338824


Verdict: MIXED. the "stale but visible" group (≥180 days since update, ≥500 impressions) only has n=17 rows out of 30,000 — way below the ~50 floor auditing-signals sets, so i can't trust a verdict from that comparison alone. on top of that, the two metrics disagree: mean ctr is lower for stale pages (matches "stale = worse"), but mean engagement_rate is higher (contradicts it).

i pulled a second, bigger table using the existing freshness_tier column to get more n. if staleness genuinely hurt performance, i'd expect ctr and engagement to decline steadily as pages get older. instead the worst ctr shows up in the middle tier (31-90 days), not the oldest one — no clean gradient. the only bucket that would really confirm the flag's logic (181+ days) is still stuck at n=17.

so: some weak, partial support when comparing the two largest tiers (0-30 vs 91-180), but no reliable, monotonic pattern, and not enough data in the extreme bucket to trust it either way. this isn't a strong enough signal to build the rule on.

### Signal check 2 — CTR vs position

claim: among visible pages ranking well, a real, sizable share fall under the 0.5% CTR threshold the real `low_ctr_visible_page` flag uses — not just a rare edge case at good positions.


In [3]:
position_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep', 'no_data']

visible['low_ctr'] = visible['ctr'] < 0.5

bucket_ctr_position = visible.groupby('position_tier').agg(
    n=('content_id', 'count'),
    mean_ctr=('ctr', 'mean'),
    share_low_ctr=('low_ctr', 'mean'),
).reindex(position_order).dropna(how='all')

bucket_ctr_position


,n,mean_ctr,share_low_ctr
position_tier,,,
top_3,458.0,0.346572,0.751092
page_1,7064.0,0.338808,0.791053
striking,4485.0,0.266798,0.849944
page_3_5,4330.0,0.143236,0.950346
deep,389.0,0.043213,0.992288


**Verdict: CONFIRMED.** ctr drops in a clean, monotonic line as position gets worse (34.7% → 33.9% → 26.7% → 14.3% → 4.3% mean ctr from top_3 to deep), and every tier has solid n (389 to 7,064 rows) — so position genuinely predicts ctr here.

but the flag's fixed 0.5% threshold isn't a rare edge case anywhere — 75.1% of pages are under it even at top_3, climbing to 99.2% at deep. so "low ctr" by itself doesn't separate a specific underperforming subgroup, it flags almost everyone regardless of position. for the rule to actually be useful, ctr needs to be compared against what's typical for a page's own tier, not a single fixed cutoff.


### My rule and its reason code

**Rule, in plain words:** a page is worth reviewing for CTR improvement if it's getting real search visibility (enough impressions to matter), and its click-through rate is meaningfully below what similar-position pages typically achieve.

this rule only uses the ctr-vs-position signal, not staleness — because signal check 1 came back MIXED (too little data, contradictory directions) while signal check 2 came back CONFIRMED once made tier-relative. the signal check is what decided what the rule gets to lean on.

**Reason code (one):** `ctr_below_tier_median` — fires when a visible page's ctr is below 70% of its own position tier's median ctr.

**Action label:** `refresh_and_review_ctr` when flagged, `monitor` otherwise.



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
from pathlib import Path

tier_median_ctr = visible.groupby('position_tier')['ctr'].median().rename('tier_median_ctr')

df = df.merge(tier_median_ctr, on='position_tier', how='left')

df['is_visible'] = (df['impressions_90d'] >= 500).astype(int)
df['underperforms_ctr'] = (
    (df['position_tier'] != 'no_data') &
    (df['ctr'] < 0.7 * df['tier_median_ctr'])
).astype(int)

df['baseline_action_score'] = df['underperforms_ctr'] * df['is_visible'] * df['impressions_90d']

df['reason_code'] = df.apply(
    lambda r: 'ctr_below_tier_median' if r['underperforms_ctr'] and r['is_visible'] else 'not_flagged',
    axis=1
)

df['action'] = df['reason_code'].map({'ctr_below_tier_median': 'refresh_and_review_ctr'}).fillna('monitor')

df['baseline_rank'] = df['baseline_action_score'].rank(method='first', ascending=False).astype(int)

output_columns = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_action_score',
    'position_tier', 'ctr', 'tier_median_ctr', 'impressions_90d',
    'reason_code', 'action',
]
queue = df[output_columns].sort_values('baseline_rank')

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print(f"wrote {len(queue)} rows to {output_path.resolve()}")
print(f"flagged (ctr_below_tier_median): {(df['reason_code'] == 'ctr_below_tier_median').sum()}")
queue.head(10)


wrote 30000 rows to C:\Users\marya\Desktop\Machine-learning-internship\work\outputs\baseline_action_score.csv
flagged (ctr_below_tier_median): 6058


,content_id,client_id,baseline_rank,baseline_action_score,position_tier,ctr,tier_median_ctr,impressions_90d,reason_code,action
6653,content_5fe46e04994d,client_4e07408562,1,517715,page_1,0.14,0.24,517715,ctr_below_tier_median,refresh_and_review_ctr
26531,content_cb112fce36be,client_19581e27de,2,309910,page_1,0.16,0.24,309910,ctr_below_tier_median,refresh_and_review_ctr
3394,content_36ff89c8214e,client_19581e27de,3,295097,page_1,0.05,0.24,295097,ctr_below_tier_median,refresh_and_review_ctr
26798,content_b28d1efd668f,client_6208ef0f77,4,286608,page_3_5,0.06,0.09,286608,ctr_below_tier_median,refresh_and_review_ctr
7678,content_8451fc6f034d,client_d029fa3a95,5,272144,top_3,0.03,0.20,272144,ctr_below_tier_median,refresh_and_review_ctr
23767,content_813e88069237,client_6208ef0f77,6,233561,page_3_5,0.06,0.09,233561,ctr_below_tier_median,refresh_and_review_ctr
26304,content_ff94c9b6b411,client_349c41201b,7,228566,page_3_5,0.04,0.09,228566,ctr_below_tier_median,refresh_and_review_ctr
6903,content_c84a0ab98e90,client_f369cb89fc,8,223271,page_1,0.03,0.24,223271,ctr_below_tier_median,refresh_and_review_ctr
15968,content_66b4046cc144,client_7f2253d7e2,9,217415,page_3_5,0.03,0.09,217415,ctr_below_tier_median,refresh_and_review_ctr
22028,content_73c54f78c06a,client_f369cb89fc,10,213963,page_1,0.10,0.24,213963,ctr_below_tier_median,refresh_and_review_ctr


In [5]:
import json

metadata = {
    "rows": int(len(queue)),
    "flagged_rows": int((df['reason_code'] == 'ctr_below_tier_median').sum()),
    "top_score": float(queue['baseline_action_score'].max()),
    "median_score_flagged": float(
        df.loc[df['reason_code'] == 'ctr_below_tier_median', 'baseline_action_score'].median()
    ),
    "reason_code": "ctr_below_tier_median",
    "rule_threshold": "ctr < 0.7 * tier_median_ctr, AND impressions_90d >= 500",
    "signal_checks": {
        "staleness": "MIXED (n=17 in strict bucket, contradictory direction, no monotonic pattern in freshness_tier breakdown)",
        "ctr_vs_position": "CONFIRMED (monotonic ctr decline by position tier, but fixed threshold not selective -- used tier-relative version instead)",
    },
}

metadata_path = output_dir / "baseline_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"wrote metadata to {metadata_path.resolve()}")
metadata


wrote metadata to C:\Users\marya\Desktop\Machine-learning-internship\work\outputs\baseline_metadata.json


{'rows': 30000,
 'flagged_rows': 6058,
 'top_score': 517715.0,
 'median_score_flagged': 2221.0,
 'reason_code': 'ctr_below_tier_median',
 'rule_threshold': 'ctr < 0.7 * tier_median_ctr, AND impressions_90d >= 500',
 'signal_checks': {'staleness': 'MIXED (n=17 in strict bucket, contradictory direction, no monotonic pattern in freshness_tier breakdown)',
  'ctr_vs_position': 'CONFIRMED (monotonic ctr decline by position tier, but fixed threshold not selective -- used tier-relative version instead)'}}

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = queue.head(20).copy()

for i,row in top20.iterrows():
    print(f"Rank {row['baseline_rank']}: content_id={row['content_id']}, client_id={row['client_id']}")
    print(f"Action: {row['action']}")
    print(f"tier_median_ctr: {row['tier_median_ctr']:.4f}")
    print(f"ctr: {row['ctr']:.4f}")
    print(f"% below tier median: {(row['tier_median_ctr'] - row['ctr']) / row['tier_median_ctr']:.2%}")
    print(f"Reason Code: {row['reason_code']}")
    print("-" * 60)

Rank 1: content_id=content_5fe46e04994d, client_id=client_4e07408562
Action: refresh_and_review_ctr
tier_median_ctr: 0.2400
ctr: 0.1400
% below tier median: 41.67%
Reason Code: ctr_below_tier_median
------------------------------------------------------------
Rank 2: content_id=content_cb112fce36be, client_id=client_19581e27de
Action: refresh_and_review_ctr
tier_median_ctr: 0.2400
ctr: 0.1600
% below tier median: 33.33%
Reason Code: ctr_below_tier_median
------------------------------------------------------------
Rank 3: content_id=content_36ff89c8214e, client_id=client_19581e27de
Action: refresh_and_review_ctr
tier_median_ctr: 0.2400
ctr: 0.0500
% below tier median: 79.17%
Reason Code: ctr_below_tier_median
------------------------------------------------------------
Rank 4: content_id=content_b28d1efd668f, client_id=client_6208ef0f77
Action: refresh_and_review_ctr
tier_median_ctr: 0.0900
ctr: 0.0600
% below tier median: 33.33%
Reason Code: ctr_below_tier_median
---------------------

1. content_5fe46e04994d — refresh_and_review_ctr — page_1 tier (median CTR 0.24), this page's CTR is 0.14 — about 42% below typical, on 517,715 impressions, the single largest page in the whole queue. What would make it wrong: if a competitor's paid ad or a rich SERP feature (reviews, image pack) sits above this result specifically, drawing clicks away regardless of title/meta quality.

2. content_cb112fce36be — refresh_and_review_ctr — page_1 tier, CTR 0.16 vs 0.24 median — about 33% below typical, 309,910 impressions. What would make it wrong: if this page recently moved up into page_1 and the CTR reflects mostly older, lower-position history — the "current" position might not match the whole averaging window.

3. content_36ff89c8214e — refresh_and_review_ctr — page_1 tier, CTR 0.05 vs 0.24 — about 79% below typical, 295,097 impressions. What would make it wrong: if this page ranks for a broad/ambiguous keyword pulling in searchers who were never going to click (informational intent landing on a transactional page).

4. content_b28d1efd668f — refresh_and_review_ctr — page_3_5 tier (median 0.09), CTR 0.06 — about 33% below typical, 286,608 impressions. What would make it wrong: if a sibling page on the same site targets a near-identical query and is absorbing the clicks this one "should" get — a consolidation issue, not a CTR issue.

5. content_8451fc6f034d — refresh_and_review_ctr — top_3 tier (median 0.20), CTR 0.03 — about 85% below typical, 272,144 impressions. What would make it wrong: if this is a branded query where users already trust the brand and navigate elsewhere directly, or a competitor's ad sits above even this top-3 result.

6. content_813e88069237 — refresh_and_review_ctr — page_3_5 tier, CTR 0.06 vs 0.09 — about 33% below typical, 233,561 impressions. What would make it wrong: if the position is volatile (bouncing between tiers during the window) and the averaged CTR doesn't reflect any single stable ranking.

7. content_ff94c9b6b411 — refresh_and_review_ctr — page_3_5 tier, CTR 0.04 vs 0.09 — about 56% below typical, 228,566 impressions. What would make it wrong: if impressions come from many unrelated long-tail queries this page happens to rank for, not the one query it was actually built for — the "audience" isn't who the median assumes.

8. content_c84a0ab98e90 — refresh_and_review_ctr — page_1 tier, CTR 0.03 vs 0.24 — about 88% below typical, 223,271 impressions, the largest gap in the top 10. What would make it wrong: if this is a data artifact — e.g. impressions logged under a redirect or duplicate URL that doesn't reflect real user-facing clicks.

9. content_66b4046cc144 — refresh_and_review_ctr — page_3_5 tier, CTR 0.03 vs 0.09 — about 67% below typical, 217,415 impressions. What would make it wrong: if a featured snippet or "People Also Ask" box directly answers the query on the results page, satisfying searchers before they ever reach this listing.

10. content_73c54f78c06a — refresh_and_review_ctr — page_1 tier, CTR 0.10 vs 0.24 — about 58% below typical, 213,963 impressions. What would make it wrong: if the page was recently retitled/relaunched and hasn't built up SERP trust yet — low CTR here could be temporary, not a lasting problem.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
weak = queue.head(10).copy()
weak['pct_below_median'] = (weak['tier_median_ctr'] - weak['ctr']) / weak['tier_median_ctr'] 

print("Weak Picks")
print("=" * 50)
weak_picks = weak[weak['pct_below_median'] < 0.40]
weak_picks[['baseline_rank', 'content_id', 'ctr', 'tier_median_ctr', 'pct_below_median']]


Weak Picks


,baseline_rank,content_id,ctr,tier_median_ctr,pct_below_median
26531,2,content_cb112fce36be,0.16,0.24,0.333333
26798,4,content_b28d1efd668f,0.06,0.09,0.333333
23767,6,content_813e88069237,0.06,0.09,0.333333


Weak picks: rows 2, 4, and 6 are the shakiest of my top 10. all three are only 33% below their tier's median ctr, barely past the rule's 30% qualifying line, not a dramatic underperformance like rows 5 or 8 (85-88% below median). they only rank this high because the score formula is flag × impressions_90d, once a page qualifies at all, the score only rewards how many impressions it has, not how badly it's actually underperforming. row 2 (33% below, 309,910 impressions) outranks row 5 (85% below, 272,144 impressions) purely because it's bigger, even though row 5's ctr problem is much more severe. this is a real weakness in the rule: it conflates "large" with "urgent," and a future version should probably weight the size of the ctr gap itself, not just treat "underperforms or not" as a yes/no switch.

Leakage check: the rule only uses ctr, position_tier, impressions_90d, and tier_median_ctr (computed from those same current-window numbers), nothing about what happens to these pages in the future, and no FlyRank product flags (like health_score or an internal priority score) were used anywhere. trend_direction/trend_pct/is_declining_label were deliberately never touched, since those are label-derived per the data dictionary. so this rule is clean on both counts.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.